
# Figure 2 generator for the ICU mortality manuscript

This notebook **does not train models**. It reads the CSV outputs generated by the combined training notebook and creates publication-style figures for the manuscript.

## Main output

**Figure 2. Predictive performance and calibration of ICU mortality prediction models**

Panels:
- **a)** ROC curves
- **b)** Precision-recall curves
- **c)** External test metric summary
- **d)** Calibration curves

## Optional outputs

If the required CSV files are present, this notebook can also create compact supplementary figures:
- **Supplementary Figure S2**: tuning diagnostics
- **Supplementary Figure S3**: feature-importance support

## Expected workflow

1. Run the model-training notebook first.
2. Confirm that the CSV outputs were written.
3. Run this notebook to generate the figures.

## Save locations

To match the manuscript draft, figures are saved into:
- `../Figure 2/`
- `../Supplementary Figures/`

The figure width is set to **170 mm** and outputs are saved as **PNG, PDF, and SVG**.


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "figure.titlesize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


In [ ]:

# ============================================================
# 1. Global configuration
# ============================================================

FIGURE_WIDTH_MM = 170
FIGURE_WIDTH_IN = FIGURE_WIDTH_MM / 25.4
MAIN_FIG_HEIGHT_IN = 5.6
SUPP_FIG_HEIGHT_IN = 5.2
SAVE_DPI = 600

MODEL_ORDER = [
    "logistic_regression",
    "xgboost_auc_optimized",
    "mlp",
]

MODEL_LABELS = {
    "logistic_regression": "Logistic Regression",
    "xgboost_auc_optimized": "XGBoost",
    "mlp": "MLP",
}

MODEL_COLORS = {
    "logistic_regression": "#4C78A8",
    "xgboost_auc_optimized": "#F58518",
    "mlp": "#54A24B",
}

METRIC_COLUMNS = ["roc_auc", "pr_auc", "f1", "brier_score"]
METRIC_LABELS = ["AUROC ↑", "AUPRC ↑", "F1 ↑", "Brier ↓"]

BASE_DIR = Path("..")
MAIN_FIG_DIR = BASE_DIR / "Figure 2"
SUPP_FIG_DIR = BASE_DIR / "Supplementary Figures"
MAIN_FIG_DIR.mkdir(parents=True, exist_ok=True)
SUPP_FIG_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR_CANDIDATES = [
    BASE_DIR / "output" / "model_training_results",
    Path("../output/model_training_results"),
    Path("./output/model_training_results"),
    Path("/mnt/data/output/model_training_results"),
]

print("Main figure directory:", MAIN_FIG_DIR.resolve())
print("Supplementary figure directory:", SUPP_FIG_DIR.resolve())


In [ ]:
# ============================================================
# 2. Helper functions
# ============================================================

def find_results_dir():
    for candidate in RESULTS_DIR_CANDIDATES:
        if candidate.exists() and (candidate / "combined_external_test_performance_summary.csv").exists():
            return candidate
    checked = "\n".join(str(p.resolve()) for p in RESULTS_DIR_CANDIDATES)
    raise FileNotFoundError(
        "Could not find model_training_results with the required CSV files. Checked:\n" + checked
    )


def panel_label(ax, label):
    ax.text(
        -0.14,
        1.08,
        label,
        transform=ax.transAxes,
        fontsize=10,
        fontweight="bold",
        va="top",
        ha="left",
    )


def save_figure(fig, out_dir, stem, dpi=SAVE_DPI):
    png_path = out_dir / f"{stem}.png"
    pdf_path = out_dir / f"{stem}.pdf"
    svg_path = out_dir / f"{stem}.svg"
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    fig.savefig(svg_path, bbox_inches="tight", facecolor="white")
    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")
    print(f"Saved: {svg_path}")


def normalize_metric_column(values, higher_is_better=True):
    arr = np.asarray(values, dtype=float)
    if np.all(np.isnan(arr)):
        return np.zeros_like(arr)
    valid = arr[~np.isnan(arr)]
    if len(valid) == 0:
        return np.zeros_like(arr)
    vmin = np.min(valid)
    vmax = np.max(valid)
    if np.isclose(vmin, vmax):
        scaled = np.ones_like(arr) * 0.6
    else:
        scaled = (arr - vmin) / (vmax - vmin)
    if not higher_is_better:
        scaled = 1 - scaled
    return scaled


def pretty_model_name(model_name):
    return MODEL_LABELS.get(model_name, model_name)


In [ ]:

# ============================================================
# 3. Load required CSV files
# ============================================================

RESULTS_DIR = find_results_dir()
print("Using results directory:", RESULTS_DIR.resolve())

perf_df = pd.read_csv(RESULTS_DIR / "combined_external_test_performance_summary.csv")
roc_df = pd.read_csv(RESULTS_DIR / "combined_roc_curve_points.csv")
pr_df = pd.read_csv(RESULTS_DIR / "combined_precision_recall_curve_points.csv")
cal_df = pd.read_csv(RESULTS_DIR / "combined_calibration_curve_points.csv")

perf_df = perf_df.copy()
perf_df["model"] = perf_df["model"].astype(str)
roc_df["model"] = roc_df["model"].astype(str)
pr_df["model"] = pr_df["model"].astype(str)
cal_df["model"] = cal_df["model"].astype(str)

perf_df = perf_df.loc[perf_df["split"] == "external_test"].copy()
perf_df["model_label"] = perf_df["model"].map(pretty_model_name)

available_models = [m for m in MODEL_ORDER if m in perf_df["model"].unique()]
print("Available models:", [pretty_model_name(m) for m in available_models])

display(perf_df[["model", "roc_auc", "pr_auc", "f1", "brier_score", "threshold"]])


In [ ]:

# ============================================================
# 4. Create Figure 2 (main manuscript figure)
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(FIGURE_WIDTH_IN, MAIN_FIG_HEIGHT_IN), constrained_layout=False)
ax_roc, ax_pr = axes[0, 0], axes[0, 1]
ax_metrics, ax_cal = axes[1, 0], axes[1, 1]

# -----------------------
# Panel a) ROC curves
# -----------------------
for model_name in available_models:
    sub = roc_df.loc[roc_df["model"] == model_name].copy()
    if sub.empty:
        continue
    auc_value = perf_df.loc[perf_df["model"] == model_name, "roc_auc"].iloc[0]
    ax_roc.plot(
        sub["fpr"],
        sub["tpr"],
        lw=1.8,
        color=MODEL_COLORS.get(model_name, "black"),
        label=f"{pretty_model_name(model_name)} (AUROC = {auc_value:.3f})",
    )

ax_roc.plot([0, 1], [0, 1], ls="--", lw=1.0, color="0.6")
ax_roc.set_xlim(0, 1)
ax_roc.set_ylim(0, 1)
ax_roc.set_xlabel("False positive rate")
ax_roc.set_ylabel("True positive rate")
ax_roc.set_title("ROC curves", pad=6)
ax_roc.legend(frameon=False, loc="lower right")
ax_roc.grid(True, alpha=0.2, linewidth=0.5)
panel_label(ax_roc, "a)")

# -----------------------
# Panel b) Precision-recall curves
# -----------------------
prevalence_values = []
for model_name in available_models:
    sub = pr_df.loc[pr_df["model"] == model_name].copy()
    if sub.empty:
        continue
    auprc_value = perf_df.loc[perf_df["model"] == model_name, "pr_auc"].iloc[0]
    ax_pr.plot(
        sub["recall"],
        sub["precision"],
        lw=1.8,
        color=MODEL_COLORS.get(model_name, "black"),
        label=f"{pretty_model_name(model_name)} (AUPRC = {auprc_value:.3f})",
    )
    if "baseline_death_rate" in sub.columns:
        prevalence_values.extend(sub["baseline_death_rate"].dropna().unique().tolist())

if prevalence_values:
    prevalence = float(np.nanmean(prevalence_values))
    ax_pr.axhline(prevalence, ls="--", lw=1.0, color="0.5", label=f"Mortality prevalence = {prevalence:.3f}")

ax_pr.set_xlim(0, 1)
ax_pr.set_ylim(0, 1)
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.set_title("Precision-recall curves", pad=6)
ax_pr.legend(frameon=False, loc="upper right")
ax_pr.grid(True, alpha=0.2, linewidth=0.5)
panel_label(ax_pr, "b)")

# -----------------------
# Panel c) Metric summary heatmap
# -----------------------
metric_table = perf_df.set_index("model").loc[available_models, METRIC_COLUMNS].copy()
metric_values = metric_table.values.astype(float)

# Normalize each metric column for shading while displaying the original values.
normalized_cols = []
for idx, metric in enumerate(METRIC_COLUMNS):
    col = metric_values[:, idx]
    higher_is_better = metric != "brier_score"
    normalized_cols.append(normalize_metric_column(col, higher_is_better=higher_is_better))
normalized_matrix = np.column_stack(normalized_cols)

cmap = LinearSegmentedColormap.from_list(
    "paper_blue",
    ["#F7FBFF", "#C6DBEF", "#6BAED6", "#2171B5"],
)

im = ax_metrics.imshow(normalized_matrix, aspect="auto", cmap=cmap, vmin=0, vmax=1)
ax_metrics.set_xticks(np.arange(len(METRIC_COLUMNS)))
ax_metrics.set_xticklabels(METRIC_LABELS)
ax_metrics.set_yticks(np.arange(len(available_models)))
ax_metrics.set_yticklabels([pretty_model_name(m) for m in available_models])
ax_metrics.set_title("External test metric summary", pad=6)

for i in range(len(available_models)):
    for j in range(len(METRIC_COLUMNS)):
        value = metric_values[i, j]
        text_color = "white" if normalized_matrix[i, j] > 0.60 else "black"
        ax_metrics.text(j, i, f"{value:.3f}", ha="center", va="center", color=text_color, fontsize=7)

for spine in ax_metrics.spines.values():
    spine.set_visible(False)
ax_metrics.set_xticks(np.arange(-0.5, len(METRIC_COLUMNS), 1), minor=True)
ax_metrics.set_yticks(np.arange(-0.5, len(available_models), 1), minor=True)
ax_metrics.grid(which="minor", color="white", linestyle='-', linewidth=1.2)
ax_metrics.tick_params(which="minor", bottom=False, left=False)
panel_label(ax_metrics, "c)")

# -----------------------
# Panel d) Calibration curves
# -----------------------
for model_name in available_models:
    sub = cal_df.loc[cal_df["model"] == model_name].copy().sort_values("mean_predicted_probability")
    if sub.empty:
        continue
    brier_value = perf_df.loc[perf_df["model"] == model_name, "brier_score"].iloc[0]
    ax_cal.plot(
        sub["mean_predicted_probability"],
        sub["observed_event_rate"],
        marker="o",
        ms=3.0,
        lw=1.5,
        color=MODEL_COLORS.get(model_name, "black"),
        label=f"{pretty_model_name(model_name)} (Brier = {brier_value:.3f})",
    )

ax_cal.plot([0, 1], [0, 1], ls="--", lw=1.0, color="0.6", label="Perfect calibration")
ax_cal.set_xlim(0, 1)
ax_cal.set_ylim(0, 1)
ax_cal.set_xlabel("Mean predicted mortality risk")
ax_cal.set_ylabel("Observed mortality rate")
ax_cal.set_title("Calibration curves", pad=6)
ax_cal.legend(frameon=False, loc="upper left")
ax_cal.grid(True, alpha=0.2, linewidth=0.5)
panel_label(ax_cal, "d)")

# Layout and save
plt.subplots_adjust(left=0.09, right=0.985, top=0.96, bottom=0.09, wspace=0.28, hspace=0.34)

main_stem = "Figure2_model_performance_calibration_170mm"
save_figure(fig, MAIN_FIG_DIR, main_stem)
plt.show()


In [ ]:

# Optional: save the exact source data tables used in Figure 2 into the same figure folder.

perf_df.to_csv(MAIN_FIG_DIR / "Figure2_source_data_performance.csv", index=False)
roc_df.to_csv(MAIN_FIG_DIR / "Figure2_source_data_roc.csv", index=False)
pr_df.to_csv(MAIN_FIG_DIR / "Figure2_source_data_pr.csv", index=False)
cal_df.to_csv(MAIN_FIG_DIR / "Figure2_source_data_calibration.csv", index=False)

print("Saved Figure 2 source-data CSVs into:", MAIN_FIG_DIR.resolve())



## Optional supplementary figure generation

The next cells are optional. They create compact supplementary figures **only if the required CSV files already exist**.


In [ ]:

# ============================================================
# 5. Optional supplementary figure S2: tuning diagnostics
# ============================================================

xgb_cv_path = RESULTS_DIR / "xgboost_auc_cv_results.csv"
mlp_tuning_path = RESULTS_DIR / "mlp_tuning_results.csv"
mlp_history_path = RESULTS_DIR / "mlp_tuning_history_all_configs.csv"

if xgb_cv_path.exists() and mlp_tuning_path.exists():
    xgb_cv = pd.read_csv(xgb_cv_path)
    mlp_tuning = pd.read_csv(mlp_tuning_path)

    fig, axes = plt.subplots(1, 2, figsize=(FIGURE_WIDTH_IN, SUPP_FIG_HEIGHT_IN * 0.65), constrained_layout=False)
    ax1, ax2 = axes

    # Panel a) top XGBoost CV settings
    xgb_rank_col = "rank_test_score" if "rank_test_score" in xgb_cv.columns else None
    xgb_mean_col = "mean_test_score" if "mean_test_score" in xgb_cv.columns else None
    if xgb_mean_col is not None:
        xgb_top = xgb_cv.sort_values(by=xgb_mean_col, ascending=False).head(12).copy()
        xgb_top = xgb_top.iloc[::-1]
        labels = [f"Cfg {i}" for i in range(len(xgb_top))]
        ax1.barh(labels, xgb_top[xgb_mean_col], color="#F58518", edgecolor="none")
        ax1.set_xlabel("Mean CV ROC-AUC")
        ax1.set_title("Top XGBoost CV settings", pad=6)
        for y, v in enumerate(xgb_top[xgb_mean_col]):
            ax1.text(v + 0.001, y, f"{v:.3f}", va="center", fontsize=7)
    else:
        ax1.text(0.5, 0.5, "Required XGBoost CV columns not found", ha="center", va="center")
        ax1.set_axis_off()
    panel_label(ax1, "a)")

    # Panel b) top MLP tuning settings
    mlp_score_col = None
    for candidate in ["selection_score", "val_roc_auc", "score"]:
        if candidate in mlp_tuning.columns:
            mlp_score_col = candidate
            break

    if mlp_score_col is not None:
        mlp_top = mlp_tuning.sort_values(by=mlp_score_col, ascending=False).head(12).copy()
        mlp_top = mlp_top.iloc[::-1]
        labels = [f"Cfg {int(cfg)}" if "config_id" in mlp_top.columns else f"Cfg {i}" for i, cfg in enumerate(mlp_top.get("config_id", pd.Series(range(len(mlp_top)))).tolist())]
        ax2.barh(labels, mlp_top[mlp_score_col], color="#54A24B", edgecolor="none")
        ax2.set_xlabel(mlp_score_col.replace("_", " ").title())
        ax2.set_title("Top MLP tuning settings", pad=6)
        for y, v in enumerate(mlp_top[mlp_score_col]):
            ax2.text(v + 0.001, y, f"{v:.3f}", va="center", fontsize=7)
    else:
        ax2.text(0.5, 0.5, "Required MLP tuning columns not found", ha="center", va="center")
        ax2.set_axis_off()
    panel_label(ax2, "b)")

    for ax in axes:
        ax.grid(True, axis="x", alpha=0.2, linewidth=0.5)
        ax.spines[["top", "right"]].set_visible(False)

    plt.subplots_adjust(left=0.10, right=0.99, top=0.92, bottom=0.12, wspace=0.35)
    supp2_stem = "Supplementary_FigureS2_model_tuning_diagnostics_170mm"
    save_figure(fig, SUPP_FIG_DIR, supp2_stem)
    plt.show()
else:
    print("Skipping Supplementary Figure S2 because the required tuning CSV files were not found.")


In [ ]:

# ============================================================
# 6. Optional supplementary figure S3: feature-importance support
# ============================================================

lr_coef_path = RESULTS_DIR / "logistic_regression_standardized_coefficients.csv"
xgb_imp_path = RESULTS_DIR / "xgboost_auc_optimized_native_feature_importance.csv"
xgb_shap_path = RESULTS_DIR / "xgboost_auc_optimized_mean_abs_shap.csv"
mlp_perm_path = RESULTS_DIR / "mlp_permutation_importance_auc_drop.csv"

available_panels = []
if lr_coef_path.exists():
    available_panels.append(("lr_coef", lr_coef_path))
if xgb_imp_path.exists():
    available_panels.append(("xgb_imp", xgb_imp_path))
if xgb_shap_path.exists():
    available_panels.append(("xgb_shap", xgb_shap_path))
if mlp_perm_path.exists():
    available_panels.append(("mlp_perm", mlp_perm_path))

if len(available_panels) >= 2:
    n_panels = min(4, len(available_panels))
    ncols = 2
    nrows = int(np.ceil(n_panels / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(FIGURE_WIDTH_IN, SUPP_FIG_HEIGHT_IN), constrained_layout=False)
    axes = np.atleast_1d(axes).ravel()

    panel_map = {
        "lr_coef": ("Logistic Regression coefficients", "abs_coefficient", "#4C78A8", "feature"),
        "xgb_imp": ("XGBoost native importance", "xgb_feature_importance", "#F58518", "feature"),
        "xgb_shap": ("XGBoost mean |SHAP|", "mean_abs_shap", "#F58518", "feature"),
        "mlp_perm": ("MLP permutation importance", "importance_mean_auc_drop", "#54A24B", "feature"),
    }

    for idx, (key, path) in enumerate(available_panels[:4]):
        ax = axes[idx]
        title, value_col, color, feature_col = panel_map[key]
        df = pd.read_csv(path)
        if value_col not in df.columns:
            ax.text(0.5, 0.5, f"Column '{value_col}' not found", ha="center", va="center")
            ax.set_axis_off()
            continue

        top = df.sort_values(by=value_col, ascending=False).head(15).copy()
        top = top.iloc[::-1]
        ax.barh(top[feature_col], top[value_col], color=color, edgecolor="none")
        ax.set_title(title, pad=6)
        ax.set_xlabel(value_col.replace("_", " "))
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(True, axis="x", alpha=0.2, linewidth=0.5)
        panel_label(ax, chr(ord('a') + idx) + ")")

    for j in range(len(available_panels[:4]), len(axes)):
        axes[j].set_axis_off()

    plt.subplots_adjust(left=0.18, right=0.99, top=0.94, bottom=0.08, wspace=0.45, hspace=0.45)
    supp3_stem = "Supplementary_FigureS3_feature_importance_support_170mm"
    save_figure(fig, SUPP_FIG_DIR, supp3_stem)
    plt.show()
else:
    print("Skipping Supplementary Figure S3 because not enough feature-importance CSV files were found.")
